# AnyProjector Phase 3 — LoRA Tool-Calling

**Architecture:** Whisper (frozen) → Q-Former Projector v0.9.7 (frozen) → Qwen2.5-1.5B (LoRA)

Phase 2 aligned projector output to text embedding space (cos_sim ≈ 0.75).  
Phase 3 teaches the LLM to **read** those 64 audio tokens and generate tool-calling responses.


In [ ]:
!pip install -q transformers datasets torch accelerate hf_transfer huggingface_hub peft bitsandbytes
import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
import torch
print(f'CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

In [ ]:
from huggingface_hub import login
login()

## Config

In [ ]:
# ═══════════════════════════════════════════
# Phase 3 Config
# ═══════════════════════════════════════════
ENCODER_ID  = "openai/whisper-medium"
LLM_ID      = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID  = "Niem/speech-massive-vie-tool-calling"

# Projector checkpoint (Phase 2 v0.9.7)
PROJECTOR_CKPT = "/content/drive/MyDrive/AnyProjector/checkpoints/phase2/v097_antiplateau/projector_best.pt"

# LoRA
LORA_RANK   = 16
LORA_ALPHA  = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "v_proj"]

# Training
BATCH_SIZE  = 4
GRAD_ACCUM  = 4   # effective = 16
LR          = 2e-4
NUM_EPOCHS  = 20
WARMUP_RATIO = 0.05
PATIENCE    = 5
VAL_SPLIT   = 0.1
MAX_TEXT_TOKENS = 256

# Projector config (must match Phase 2)
NUM_QUERIES    = 64
QFORMER_DIM    = 768
QFORMER_LAYERS = 4
QFORMER_HEADS  = 16
PROJ_DROPOUT   = 0.1  # v0.9.7 trained with dropout=0.1

SAVE_DIR = "checkpoints/phase3/lora_tool_calling"
DRIVE_DIR = "/content/drive/MyDrive/AnyProjector/checkpoints/phase3/lora_tool_calling"
SAMPLE_RATE = 16000

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DRIVE_DIR, exist_ok=True)

## Load Dataset

In [ ]:
from datasets import load_dataset

print(f"Loading {DATASET_ID}...")
raw_ds = load_dataset(DATASET_ID)
print(f"Splits: {list(raw_ds.keys())}")

# Check columns
first_split = list(raw_ds.keys())[0]
print(f"Columns: {raw_ds[first_split].column_names}")
print(f"Total: {sum(len(raw_ds[s]) for s in raw_ds)}")
print(f"Sample: {raw_ds[first_split][0]}")

## Q-Former Projector (v0.9.7 — with Dropout)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class QFormerLayer(nn.Module):
    """Q-Former layer with dropout (v0.9.7+)."""
    def __init__(self, qformer_dim, encoder_dim, num_heads=8, ffn_ratio=4, dropout=0.0):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=qformer_dim, num_heads=num_heads, batch_first=True)
        self.self_attn_norm = nn.LayerNorm(qformer_dim)
        self.self_attn_drop = nn.Dropout(dropout)
        self.cross_attn = nn.MultiheadAttention(embed_dim=qformer_dim, num_heads=num_heads, kdim=encoder_dim, vdim=encoder_dim, batch_first=True)
        self.cross_attn_norm = nn.LayerNorm(qformer_dim)
        self.cross_attn_drop = nn.Dropout(dropout)
        ffn_hidden = qformer_dim * ffn_ratio
        self.ffn = nn.Sequential(nn.Linear(qformer_dim, ffn_hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_hidden, qformer_dim))
        self.ffn_norm = nn.LayerNorm(qformer_dim)

    def forward(self, queries, encoder_out, encoder_mask=None):
        q = self.self_attn_norm(queries)
        q, _ = self.self_attn(q, q, q)
        queries = queries + self.self_attn_drop(q)
        q = self.cross_attn_norm(queries)
        q, _ = self.cross_attn(query=q, key=encoder_out, value=encoder_out, key_padding_mask=encoder_mask)
        queries = queries + self.cross_attn_drop(q)
        queries = queries + self.ffn(self.ffn_norm(queries))
        return queries

class AnyProjector(nn.Module):
    """Q-Former Projector (v0.9.7 compatible)."""
    def __init__(self, encoder_dim, llm_dim, num_queries=64, qformer_dim=768, num_layers=2, num_heads=8, dropout=0.0):
        super().__init__()
        self.encoder_dim = encoder_dim
        self.llm_dim = llm_dim
        self.num_queries = num_queries
        self.qformer_dim = qformer_dim
        self.pre_proj = nn.Sequential(nn.Linear(encoder_dim, encoder_dim), nn.GELU(), nn.LayerNorm(encoder_dim))
        self.query_tokens = nn.Parameter(torch.randn(1, num_queries, qformer_dim) * 0.02)
        self.layers = nn.ModuleList([QFormerLayer(qformer_dim, encoder_dim, num_heads, dropout=dropout) for _ in range(num_layers)])
        self.output_norm = nn.LayerNorm(qformer_dim)
        self.output_proj = nn.Sequential(nn.Linear(qformer_dim, llm_dim))

    def forward(self, encoder_output, encoder_mask=None):
        B = encoder_output.shape[0]
        encoder_output = self.pre_proj(encoder_output)
        queries = self.query_tokens.expand(B, -1, -1)
        for layer in self.layers:
            queries = layer(queries, encoder_output, encoder_mask)
        return self.output_proj(self.output_norm(queries))

    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())

print('OK: AnyProjector defined (v0.9.7 with dropout)')

## Load Models

In [ ]:
from transformers import WhisperProcessor, WhisperModel, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Whisper Encoder (FROZEN)
print("Loading Whisper encoder...")
processor = WhisperProcessor.from_pretrained(ENCODER_ID)
encoder = WhisperModel.from_pretrained(ENCODER_ID).encoder.to(device)
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False
print(f"  Encoder: frozen, {sum(p.numel() for p in encoder.parameters()):,} params")

# 2. Projector (FROZEN, v0.9.7)
print(f"Loading projector: {PROJECTOR_CKPT}")
ckpt = torch.load(PROJECTOR_CKPT, map_location=device, weights_only=False)
proj_config = ckpt.get('config', {})
proj_sd = ckpt.get('projector_state_dict', ckpt)  # support both formats

projector = AnyProjector(
    encoder_dim=proj_config.get('encoder_dim', 1024),
    llm_dim=proj_config.get('llm_dim', 1536),
    num_queries=proj_config.get('num_queries', NUM_QUERIES),
    qformer_dim=proj_config.get('qformer_dim', QFORMER_DIM),
    num_layers=proj_config.get('qformer_layers', QFORMER_LAYERS),
    num_heads=proj_config.get('qformer_heads', QFORMER_HEADS),
    dropout=proj_config.get('dropout', PROJ_DROPOUT),
)
projector.load_state_dict(proj_sd)
projector.to(device).eval()
for p in projector.parameters():
    p.requires_grad = False
print(f"  Projector: frozen, {projector.count_parameters():,} params, dropout={proj_config.get('dropout', PROJ_DROPOUT)}")
del ckpt, proj_sd

# 3. LLM + LoRA
print(f"Loading LLM: {LLM_ID}")
tokenizer = AutoTokenizer.from_pretrained(LLM_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
llm = AutoModelForCausalLM.from_pretrained(LLM_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16)

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGETS,
    lora_dropout=LORA_DROPOUT,
    bias="none", task_type="CAUSAL_LM",
)
llm = get_peft_model(llm, lora_config)
lora_params = sum(p.numel() for p in llm.parameters() if p.requires_grad)
print(f"  LLM LoRA: {lora_params:,} trainable params")

embed_layer = llm.get_base_model().get_input_embeddings()
llm_dtype = torch.bfloat16

import gc; gc.collect(); torch.cuda.empty_cache()
alloc = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"  VRAM: {alloc:.1f}/{total:.1f} GB")

## Dataset + DataLoader

In [ ]:
from torch.utils.data import Dataset, DataLoader
import numpy as np
import random

class Phase3Dataset(Dataset):
    """Audio + instruction → expected output."""
    def __init__(self, entries, sample_rate=16000, max_audio_seconds=30.0):
        self.entries = entries
        self.sr = sample_rate
        self.max_samples = int(max_audio_seconds * sample_rate)

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        row = self.entries[idx]
        # Audio
        audio = row['audio']
        if isinstance(audio, dict):
            wav = np.array(audio['array'], dtype=np.float32)
        else:
            wav = np.array(audio, dtype=np.float32)
        if len(wav) > self.max_samples:
            wav = wav[:self.max_samples]

        # Text fields — adapt to dataset columns
        # Try common column names
        instruction = row.get('instruction', row.get('utt', row.get('transcription', '')))
        output = row.get('output', row.get('response', row.get('expected_output', '')))

        return {'waveform': wav, 'instruction': str(instruction), 'output': str(output)}

def collate_phase3(batch):
    # Pad waveforms
    max_len = max(len(b['waveform']) for b in batch)
    waveforms = np.zeros((len(batch), max_len), dtype=np.float32)
    lengths = []
    for i, b in enumerate(batch):
        w = b['waveform']
        waveforms[i, :len(w)] = w
        lengths.append(len(w))
    return {
        'waveforms': torch.from_numpy(waveforms),
        'lengths': lengths,
        'instructions': [b['instruction'] for b in batch],
        'outputs': [b['output'] for b in batch],
    }

# Split dataset
if 'train' in raw_ds and 'test' in raw_ds:
    train_entries = list(raw_ds['train'])
    val_entries = list(raw_ds['test'])
elif 'train' in raw_ds and 'validation' in raw_ds:
    train_entries = list(raw_ds['train'])
    val_entries = list(raw_ds['validation'])
else:
    # Single split → manual 90/10
    all_entries = list(raw_ds[first_split])
    random.seed(42)
    random.shuffle(all_entries)
    split_idx = int(len(all_entries) * (1 - VAL_SPLIT))
    train_entries = all_entries[:split_idx]
    val_entries = all_entries[split_idx:]

train_ds = Phase3Dataset(train_entries)
val_ds = Phase3Dataset(val_entries)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_phase3, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_phase3, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')
print(f'Batches/epoch: {len(train_loader)}')

# Peek at one sample
s = train_ds[0]
print(f"Sample instruction: {s['instruction'][:100]}")
print(f"Sample output: {s['output'][:100]}")

## Trainer

In [ ]:
import logging, math, time, csv, shutil
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)
log = logging.getLogger('phase3')
log.setLevel(logging.INFO)

class Phase3Trainer:
    def __init__(self):
        self.best_val_loss = float('inf')
        self.patience_counter = 0
        self.log_data = []

    def _encode_audio(self, waveforms, lengths):
        """Audio → Whisper → Projector → audio_embeds (B, 64, llm_dim)."""
        inputs = processor(
            [w.numpy() for w in waveforms],
            sampling_rate=SAMPLE_RATE,
            return_tensors='pt', padding='max_length',
        )
        input_features = inputs.input_features.to(device)
        with torch.no_grad():
            enc_out = encoder(input_features).last_hidden_state
            audio_embeds = projector(enc_out.float())  # (B, 64, 1536)
        return audio_embeds

    def _build_inputs(self, audio_embeds, instructions, targets):
        """Build [prefix | audio | suffix | target] embeddings + labels."""
        B = audio_embeds.shape[0]

        # Prefix: system + user
        prefix_texts = [f'<|im_start|>user\nTranscribe the following audio in Vietnamese:\n' for _ in range(B)]
        tokenizer.padding_side = 'left'
        prefix_tok = tokenizer(prefix_texts, return_tensors='pt', padding=True, add_special_tokens=False).to(device)
        with torch.no_grad():
            prefix_embeds = embed_layer(prefix_tok.input_ids)
        prefix_mask = prefix_tok.attention_mask

        # Audio mask
        audio_mask = torch.ones((B, NUM_QUERIES), dtype=torch.long, device=device)

        # Suffix
        suffix_text = '<|im_end|>\n<|im_start|>assistant\n'
        tokenizer.padding_side = 'right'
        suffix_tok = tokenizer([suffix_text] * B, return_tensors='pt', padding=True, add_special_tokens=False).to(device)
        with torch.no_grad():
            suffix_embeds = embed_layer(suffix_tok.input_ids)
        suffix_mask = suffix_tok.attention_mask

        # Target
        target_texts = [f'{t}<|im_end|>' for t in targets]
        target_tok = tokenizer(
            target_texts, return_tensors='pt', padding=True,
            add_special_tokens=False, truncation=True, max_length=MAX_TEXT_TOKENS,
        ).to(device)
        with torch.no_grad():
            target_embeds = embed_layer(target_tok.input_ids)
        target_mask = target_tok.attention_mask

        # Concat all
        full_embeds = torch.cat([prefix_embeds, audio_embeds, suffix_embeds, target_embeds], dim=1).to(llm_dtype)
        full_mask = torch.cat([prefix_mask, audio_mask, suffix_mask, target_mask], dim=1)

        # Labels: -100 for everything except target
        ignore_prefix = torch.full_like(prefix_tok.input_ids, -100)
        ignore_audio = torch.full((B, NUM_QUERIES), -100, dtype=torch.long, device=device)
        ignore_suffix = torch.full_like(suffix_tok.input_ids, -100)
        target_labels = target_tok.input_ids.clone()
        target_labels[target_mask == 0] = -100

        labels = torch.cat([ignore_prefix, ignore_audio, ignore_suffix, target_labels], dim=1)

        return full_embeds, full_mask, labels

    def _forward_batch(self, batch):
        """Forward pass → loss."""
        audio_embeds = self._encode_audio(batch['waveforms'], batch['lengths'])
        full_embeds, full_mask, labels = self._build_inputs(
            audio_embeds, batch['instructions'], batch['outputs'],
        )
        outputs = llm(inputs_embeds=full_embeds, attention_mask=full_mask, labels=labels)
        return outputs.loss

    @torch.no_grad()
    def validate(self):
        llm.eval()
        total_loss, count = 0.0, 0
        for batch in val_loader:
            loss = self._forward_batch(batch)
            total_loss += loss.item() * len(batch['outputs'])
            count += len(batch['outputs'])
        llm.train()
        return total_loss / max(count, 1)

    def save_checkpoint(self, epoch, val_loss, is_best=False):
        tag = 'best' if is_best else f'epoch_{epoch}'
        local_dir = f'{SAVE_DIR}/{tag}'
        os.makedirs(local_dir, exist_ok=True)
        llm.save_pretrained(local_dir)
        log.info(f'  Saved: {local_dir}')
        # Drive backup
        try:
            drive_dir = f'{DRIVE_DIR}/{tag}'
            if os.path.exists(drive_dir):
                shutil.rmtree(drive_dir)
            shutil.copytree(local_dir, drive_dir)
            log.info(f'  -> Drive: {tag}')
        except Exception as e:
            log.warning(f'  Drive backup failed: {e}')

    def train(self):
        # Optimizer + Scheduler
        trainable = [p for p in llm.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable, lr=LR, weight_decay=0.01)
        total_steps = len(train_loader) * NUM_EPOCHS // GRAD_ACCUM
        warmup_steps = int(total_steps * WARMUP_RATIO)

        def lr_lambda(step):
            if step < warmup_steps:
                return step / max(warmup_steps, 1)
            progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
            return max(0.1, 0.5 * (1 + math.cos(math.pi * progress)))

        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

        log.info('=' * 60)
        log.info(f'  PHASE 3 LoRA TOOL-CALLING')
        log.info(f'  LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, targets={LORA_TARGETS}')
        log.info(f'  Trainable: {sum(p.numel() for p in trainable):,} params')
        log.info(f'  Train: {len(train_ds)}, Val: {len(val_ds)}')
        log.info(f'  Batch: {BATCH_SIZE}x{GRAD_ACCUM}={BATCH_SIZE*GRAD_ACCUM}, LR: {LR}')
        log.info(f'  Steps: {total_steps}, Warmup: {warmup_steps}')
        log.info('=' * 60)

        global_step = 0
        llm.train()

        for epoch in range(1, NUM_EPOCHS + 1):
            t0 = time.time()
            epoch_loss, batch_count = 0.0, 0
            optimizer.zero_grad()

            for step, batch in enumerate(train_loader):
                loss = self._forward_batch(batch)
                loss_scaled = loss / GRAD_ACCUM
                loss_scaled.backward()

                epoch_loss += loss.item()
                batch_count += 1

                if (step + 1) % GRAD_ACCUM == 0:
                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

            # Remaining grads
            if batch_count % GRAD_ACCUM != 0:
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            train_loss = epoch_loss / max(batch_count, 1)
            elapsed = time.time() - t0

            # Validate
            val_loss = self.validate()
            lr_now = scheduler.get_last_lr()[0]

            # Gradient diagnostics
            lora_grad = sum(p.grad.norm().item() for p in trainable if p.grad is not None)

            # Check improvement
            improved = val_loss < self.best_val_loss
            tag = ''
            if improved:
                self.best_val_loss = val_loss
                self.patience_counter = 0
                self.save_checkpoint(epoch, val_loss, is_best=True)
                tag = ' BEST'
            else:
                self.patience_counter += 1

            log.info(f'  Epoch {epoch:2d} | Train={train_loss:.4f} Val={val_loss:.4f}{tag}')
            log.info(f'           | LR={lr_now:.2e} Grad={lora_grad:.4f} | {elapsed:.0f}s')
            log.info(f'           | Patience: {self.patience_counter}/{PATIENCE}')

            # Log
            self.log_data.append({
                'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
                'lr': lr_now, 'grad_norm': lora_grad, 'elapsed_s': elapsed,
            })

            # Early stopping
            if self.patience_counter >= PATIENCE:
                log.info(f'  Early stopping at epoch {epoch}')
                break

        # Save training log
        log_path = f'{SAVE_DIR}/training_log.csv'
        with open(log_path, 'w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=self.log_data[0].keys())
            writer.writeheader()
            writer.writerows(self.log_data)
        try:
            shutil.copy(log_path, f'{DRIVE_DIR}/training_log.csv')
        except: pass
        log.info(f'  Training log: {log_path}')

print('OK: Phase3Trainer defined')

## Train

In [ ]:
trainer = Phase3Trainer()
trainer.train()

## Quick Eval — Generate from Audio

In [ ]:
@torch.no_grad()
def generate_from_audio(idx, max_tokens=128):
    """Generate LLM output from a val sample."""
    llm.eval()
    sample = val_ds[idx]
    wav = torch.from_numpy(sample['waveform']).unsqueeze(0)

    # Encode
    inputs = processor([wav[0].numpy()], sampling_rate=SAMPLE_RATE, return_tensors='pt', padding='max_length')
    enc_out = encoder(inputs.input_features.to(device)).last_hidden_state
    audio_embeds = projector(enc_out.float())

    # Build prompt
    prefix = '<|im_start|>user\nTranscribe the following audio in Vietnamese:\n'
    suffix = '<|im_end|>\n<|im_start|>assistant\n'
    prefix_ids = tokenizer(prefix, return_tensors='pt', add_special_tokens=False).input_ids.to(device)
    suffix_ids = tokenizer(suffix, return_tensors='pt', add_special_tokens=False).input_ids.to(device)
    prefix_embeds = embed_layer(prefix_ids)
    suffix_embeds = embed_layer(suffix_ids)

    input_embeds = torch.cat([prefix_embeds, audio_embeds, suffix_embeds], dim=1).to(llm_dtype)
    attn_mask = torch.ones(1, input_embeds.shape[1], dtype=torch.long, device=device)

    outputs = llm.generate(
        inputs_embeds=input_embeds, attention_mask=attn_mask,
        max_new_tokens=max_tokens, do_sample=False,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
    )
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f'--- Sample {idx} ---')
    print(f'Instruction: {sample["instruction"][:100]}')
    print(f'Expected:    {sample["output"][:200]}')
    print(f'Generated:   {generated[:200]}')
    print()

# Test on 5 val samples
for i in range(min(5, len(val_ds))):
    generate_from_audio(i)